# VAE Advanced Topics

## 📚 Learning Objectives

By completing this notebook, you will:
- Use β-VAE, disentanglement, or hierarchical VAEs
- Analyze latent structure and metrics

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

## Official Structure Reference

This notebook supports **Course 10, Unit 3** requirements from `DETAILED_UNIT_DESCRIPTIONS.md`.

---


# VAE Advanced Topics
## AIAT 124 - Generative AI

---

## 📚 Learning Objectives | أهداف التعلم

This notebook demonstrates key concepts through hands-on examples.

By completing this notebook, you will:
- Understand advanced VAE variants
- Work with conditional VAEs
- Explore latent space interpolation
- Apply VAEs to complex data

---

## 🔗 Prerequisites | المتطلبات الأساسية

- ✅ Python 3.8+ installed
- ✅ Required libraries (see `requirements.txt`)
- ✅ Basic Python knowledge

---

## Real-World Context

You are using advanced VAE techniques for generating diverse fashion designs with specific style attributes.

---


## Conditional VAE

Implement conditional VAE for controlled generation.


## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import torch
import torch.nn as nn

# Conditional VAE structure
class ConditionalVAE(nn.Module):
    def __init__(self, input_dim, latent_dim, condition_dim):
        super().__init__()
        # TODO: Define encoder and decoder with condition
        pass
    
    def encode(self, x, condition):
        # TODO: Encode with condition
        pass
    
    def decode(self, z, condition):
        # TODO: Decode with condition
        pass

print('Conditional VAE defined')

## Latent Space Interpolation

Interpolate between points in latent space.


In [ ]:
def interpolate_latent(z1, z2, n_steps=10):
    """Interpolate between two latent vectors."""
    pass


## 🌍 Real-World Worked Example — Anomaly Detection with VAE

**Industry context:**
- **Manufacturing:** Bosch uses VAEs to detect defective car parts on assembly lines
- **Finance:** PayPal uses VAEs to detect fraudulent transactions
- **Healthcare:** VAEs detect anomalous MRI scans

We train a VAE on **normal MNIST digits** then use **reconstruction error** to flag anomalies (digits the VAE has never seen).

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt

torch.manual_seed(42)
transform = T.Compose([T.ToTensor()])
dataset   = torchvision.datasets.MNIST('/tmp/mnist', train=True, download=True, transform=transform)
# Train only on digit "0" (normal class)
idx_0 = [i for i,(x,y) in enumerate(dataset) if y==0][:2000]
normal_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(dataset, idx_0), batch_size=64, shuffle=True)

# ── VAE ──────────────────────────────────────────────────────────────────
class VAE(nn.Module):
    def __init__(self, z=16):
        super().__init__()
        self.enc_fc = nn.Sequential(nn.Flatten(), nn.Linear(784,256), nn.ReLU())
        self.mu     = nn.Linear(256, z)
        self.logvar = nn.Linear(256, z)
        self.dec_fc = nn.Sequential(nn.Linear(z,256), nn.ReLU(), nn.Linear(256,784), nn.Sigmoid())
    def encode(self, x):
        h = self.enc_fc(x)
        return self.mu(h), self.logvar(h)
    def reparameterise(self, mu, lv):
        return mu + (0.5*lv).exp() * torch.randn_like(mu)
    def decode(self, z): return self.dec_fc(z).view(-1,1,28,28)
    def forward(self, x):
        mu, lv = self.encode(x)
        z = self.reparameterise(mu, lv)
        return self.decode(z), mu, lv

model = VAE(); opt = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(15):
    total=0
    for x,_ in normal_loader:
        recon, mu, lv = model(x)
        recon_loss = nn.functional.binary_cross_entropy(recon, x, reduction='sum')
        kl_loss    = -0.5 * (1 + lv - mu.pow(2) - lv.exp()).sum()
        loss = recon_loss + 0.5*kl_loss
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    if epoch%5==0: print(f"Epoch {epoch} — loss: {total/len(idx_0):.2f}")

# ── Anomaly detection: digit 0 (normal) vs digit 8 (anomaly) ──────────────
model.eval()
test_0 = torchvision.datasets.MNIST('/tmp/mnist', train=False, transform=transform)
samples_0 = torch.stack([test_0[i][0] for i in range(50) if test_0[i][1]==0])
samples_8 = torch.stack([test_0[i][0] for i in range(200) if test_0[i][1]==8][:50])

def recon_error(imgs):
    with torch.no_grad():
        recon,_,_ = model(imgs)
        return nn.functional.mse_loss(recon, imgs, reduction='none').view(len(imgs),-1).mean(1)

err_0 = recon_error(samples_0).numpy()
err_8 = recon_error(samples_8).numpy()
print(f"\nReconstruction error — Normal (0): {err_0.mean():.4f}  Anomaly (8): {err_8.mean():.4f}")
print("Higher error = anomaly detected ✅  (Same principle PayPal uses for fraud detection)")

plt.figure(figsize=(8,3))
plt.hist(err_0, bins=20, alpha=0.6, label="Normal (digit 0)")
plt.hist(err_8, bins=20, alpha=0.6, label="Anomaly (digit 8)")
plt.axvline(err_0.mean()+2*err_0.std(), color='red', linestyle='--', label="Threshold")
plt.legend(); plt.title("VAE Anomaly Detection — Production Pattern"); plt.tight_layout(); plt.show()

## 📚 References & Further Reading

**Papers:**
- Kingma & Welling (2014) — [VAE: Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) *(foundational)*
- Higgins et al. (2017) — [beta-VAE: Learning Basic Visual Concepts](https://openreview.net/forum?id=Sy2fchgcx)

**Applications:**
- Anomaly detection in manufacturing (Siemens, Bosch)
- Drug molecule generation (Insilico Medicine)

**State-of-the-Art:** Stable Diffusion's latent space is a VAE-encoded image space.

## 📝 Summary

You explored advanced **Variational Autoencoder (VAE)** techniques including disentangled representations and β-VAE. A well-trained VAE learns a structured latent space where interpolation produces meaningful results. Used in drug discovery, anomaly detection, and controllable image generation.